In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline as imbPipeline
from feature_engine import encoding as ce
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier

c:\Users\zhcri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\experimental\enable_hist_gradient_boosting.py:18: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [4]:
df=pd.read_csv('train_apps.csv',sep=',')
df.head()

,front_id,decision_day,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,corp_credit_products,sum_deb_ul_90,sum_deb_ul_30,...,overdraft_app_term_max_360,days_from_authperson_registration,fl_hdb_bki_total_active_products,corp_list,count_all_corp_dashboard_events,p75_time_spent_minutes,sum_deb_investment_90,db_group_last,fl_adminarea,target_value
0,127345,2024-02-01,1.339991,-1.847954,-1.586546,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lombard,NaN,0
1,127209,2024-02-01,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,0.842771,NaN,NaN,...,NaN,NaN,NaN,-0.862289,-3.400318,-0.780786,NaN,inn_scoring,NaN,0
2,272776,2024-02-01,2.185431,3.167063,2.369547,-0.709770,-0.400695,0.000000,0.834373,4.897257,...,NaN,NaN,NaN,1.168810,3.015012,0.554064,NaN,NaN,NaN,0
3,127210,2024-02-01,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,164500,2024-02-01,0.845440,4.559196,3.467730,-2.484194,-0.400695,0.000000,-0.518122,-3.435251,...,NaN,1.067447,NaN,1.022327,1.506380,0.190096,NaN,NaN,NaN,0


In [12]:
for col in df.columns:
    print(f" {col} - {df[col].isnull().sum()/df.shape[0]*100}")

 front_id - 0.0
 decision_day - 0.0
 loan_amount_last - 0.0
 overdraft_limit_min - 0.0
 overdraft_limit_max - 0.0
 offered_rate - 0.0
 cb_rate - 0.0
 corp_credit_products - 35.243491851474445
 sum_deb_ul_90 - 37.25600897818109
 sum_deb_ul_30 - 42.311055418235895
 cnt_deb_loan_90 - 21.65917337390957
 cnt_deb_ul_ip_90 - 20.86532039851006
 cnt_deb_ul_ip_30 - 22.9776715941091
 balance_rur_amt_30_min - 23.97807781549287
 cnt_cred_loan_90 - 21.65917337390957
 loan_rev_max_start_non_fin - 91.32063260374137
 loan_rev_min_start_fin - 85.86143031237737
 app_term_mean_360 - 38.47605015112813
 overdraft_app_term_max_360 - 96.20699389290903
 days_from_authperson_registration - 54.02950957374295
 fl_hdb_bki_total_active_products - 16.777631660481546
 corp_list - 35.243491851474445
 count_all_corp_dashboard_events - 35.243491851474445
 p75_time_spent_minutes - 35.243491851474445
 sum_deb_investment_90 - 88.60927699478798
 db_group_last - 38.47605015112813
 fl_adminarea - 29.819403611927765
 target_va

In [23]:
df_feature=df.drop(columns=['target_value'],axis=1)
df_feature

,front_id,decision_day,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,corp_credit_products,sum_deb_ul_90,sum_deb_ul_30,...,app_term_mean_360,overdraft_app_term_max_360,days_from_authperson_registration,fl_hdb_bki_total_active_products,corp_list,count_all_corp_dashboard_events,p75_time_spent_minutes,sum_deb_investment_90,db_group_last,fl_adminarea
0,127345,2024-02-01,1.339991,-1.847954,-1.586546,1.774424,-0.400695,NaN,NaN,NaN,...,1.767094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lombard,NaN
1,127209,2024-02-01,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,0.842771,NaN,NaN,...,-0.504888,NaN,NaN,NaN,-0.862289,-3.400318,-0.780786,NaN,inn_scoring,NaN
2,272776,2024-02-01,2.185431,3.167063,2.369547,-0.709770,-0.400695,0.000000,0.834373,4.897257,...,NaN,NaN,NaN,NaN,1.168810,3.015012,0.554064,NaN,NaN,NaN
3,127210,2024-02-01,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,164500,2024-02-01,0.845440,4.559196,3.467730,-2.484194,-0.400695,0.000000,-0.518122,-3.435251,...,NaN,NaN,1.067447,NaN,1.022327,1.506380,0.190096,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145236,258918,2025-06-05,1.339991,2.480726,1.828130,3.903733,1.602779,0.000000,0.893285,5.247350,...,0.000000,NaN,2.592372,1.302069,2.628996,1.589065,0.706500,NaN,zalog_light,Новосибирская область
145237,176413,2025-06-05,0.845440,3.639458,3.605443,1.596982,1.602779,1.229059,0.576790,2.776671,...,0.000000,NaN,-0.384331,-4.700790,1.214168,0.665258,0.306940,NaN,inn_scoring,г. Москва
145238,258920,2025-06-05,1.690880,3.960429,2.995393,3.903733,1.602779,1.234158,0.691625,4.440577,...,0.000000,NaN,1.834869,-1.351879,1.806117,2.272156,0.647315,NaN,inn_scoring,г. Санкт - Петербург
145239,174837,2025-06-05,0.845440,-2.742346,-2.292086,2.661636,1.602779,1.541025,0.109289,0.918909,...,-1.678739,1.86909,-1.594352,1.674455,1.050198,-0.305594,0.394573,-0.105689,inn_scoring,Красноярский край


In [24]:
df_feature['cnt_deb_ul_ip_90'] = df_feature['cnt_deb_ul_ip_90'].fillna(df_feature['cnt_deb_ul_ip_90'].median())
df_feature['cnt_cred_loan_90_flag']=df_feature['cnt_cred_loan_90'].isna().astype(int)
df_feature['cnt_cred_loan_90'] = df_feature['cnt_cred_loan_90'].fillna(df_feature['cnt_cred_loan_90'].median())
df_feature['fl_hdb_bki_total_active_products'] = df_feature['fl_hdb_bki_total_active_products'].fillna(df_feature['fl_hdb_bki_total_active_products'].median())
df_feature['balance_rur_amt_30_min_flag']=df_feature['balance_rur_amt_30_min'].isna().astype(int)
df_feature['balance_rur_amt_30_min']=df_feature['balance_rur_amt_30_min'].fillna(0)

In [25]:
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] < -50,-9999)
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] > 50,9999)

In [26]:
df_overdraft = df_feature.loc[df_feature['db_group_last'] == 'overdraft',['overdraft_app_term_max_360']]

In [27]:

df_feature['count_all_corp_dashboard_events'] = df_feature['count_all_corp_dashboard_events'].fillna(0)

In [28]:
df_feature=df_feature.drop(columns=['front_id','decision_day',
                                    'corp_credit_products','sum_deb_ul_90',
                                    'sum_deb_ul_30','cnt_deb_loan_90',
                                    'sum_deb_ul_90','cnt_deb_ul_ip_30',
                                    'loan_rev_max_start_non_fin','loan_rev_min_start_fin',
                                    'app_term_mean_360','overdraft_app_term_max_360','days_from_authperson_registration',
                                    'corp_list',
                                    'p75_time_spent_minutes','sum_deb_investment_90'],axis=1)

In [29]:
df_feature

,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,cnt_deb_ul_ip_90,balance_rur_amt_30_min,cnt_cred_loan_90,fl_hdb_bki_total_active_products,count_all_corp_dashboard_events,db_group_last,fl_adminarea,cnt_cred_loan_90_flag,balance_rur_amt_30_min_flag
0,1.339991,-1.847954,-1.586546,1.774424,-0.400695,0.147288,-5.561599,0.000000,0.000000,0.000000,lombard,NaN,1,0
1,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,-4.466656,1.466815,0.000000,0.000000,-3.400318,inn_scoring,NaN,0,0
2,2.185431,3.167063,2.369547,-0.709770,-0.400695,5.291707,4.041974,0.000000,0.000000,3.015012,NaN,NaN,0,0
3,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,0.147288,-3.322507,0.000000,0.000000,0.000000,NaN,NaN,1,0
4,0.845440,4.559196,3.467730,-2.484194,-0.400695,-1.373655,4.324315,6.729738,0.000000,1.506380,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145236,1.339991,2.480726,1.828130,3.903733,1.602779,2.599715,-5.527199,0.000000,1.302069,1.589065,zalog_light,Новосибирская область,0,0
145237,0.845440,3.639458,3.605443,1.596982,1.602779,-1.037194,3.052725,0.000000,-4.700790,0.665258,inn_scoring,г. Москва,0,0
145238,1.690880,3.960429,2.995393,3.903733,1.602779,2.863267,-5.561599,8.491984,-1.351879,2.272156,inn_scoring,г. Санкт - Петербург,0,0
145239,0.845440,-2.742346,-2.292086,2.661636,1.602779,0.238218,-5.561599,16.983968,1.674455,-0.305594,inn_scoring,Красноярский край,0,0


In [30]:
df_feature

,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,cnt_deb_ul_ip_90,balance_rur_amt_30_min,cnt_cred_loan_90,fl_hdb_bki_total_active_products,count_all_corp_dashboard_events,db_group_last,fl_adminarea,cnt_cred_loan_90_flag,balance_rur_amt_30_min_flag
0,1.339991,-1.847954,-1.586546,1.774424,-0.400695,0.147288,-5.561599,0.000000,0.000000,0.000000,lombard,NaN,1,0
1,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,-4.466656,1.466815,0.000000,0.000000,-3.400318,inn_scoring,NaN,0,0
2,2.185431,3.167063,2.369547,-0.709770,-0.400695,5.291707,4.041974,0.000000,0.000000,3.015012,NaN,NaN,0,0
3,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,0.147288,-3.322507,0.000000,0.000000,0.000000,NaN,NaN,1,0
4,0.845440,4.559196,3.467730,-2.484194,-0.400695,-1.373655,4.324315,6.729738,0.000000,1.506380,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145236,1.339991,2.480726,1.828130,3.903733,1.602779,2.599715,-5.527199,0.000000,1.302069,1.589065,zalog_light,Новосибирская область,0,0
145237,0.845440,3.639458,3.605443,1.596982,1.602779,-1.037194,3.052725,0.000000,-4.700790,0.665258,inn_scoring,г. Москва,0,0
145238,1.690880,3.960429,2.995393,3.903733,1.602779,2.863267,-5.561599,8.491984,-1.351879,2.272156,inn_scoring,г. Санкт - Петербург,0,0
145239,0.845440,-2.742346,-2.292086,2.661636,1.602779,0.238218,-5.561599,16.983968,1.674455,-0.305594,inn_scoring,Красноярский край,0,0


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint


num_features = [
    'loan_amount_last','overdraft_limit_min',
    'overdraft_limit_max','offered_rate',
    'cb_rate','cnt_deb_ul_ip_90','balance_rur_amt_30_min',
    'cnt_cred_loan_90','fl_hdb_bki_total_active_products','count_all_corp_dashboard_events'
]

cat_features = ['fl_adminarea', 'db_group_last']

flag_features = [
    'balance_rur_amt_30_min_flag',
    'cnt_cred_loan_90_flag'
]

X = df_feature[num_features + cat_features + flag_features]
y = df['target_value']


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('flag', 'passthrough', flag_features)
])


pipe = Pipeline([
    ('prep', preprocess),
    ('model', HistGradientBoostingClassifier(
        loss='log_loss',
        random_state=42,
        early_stopping=True,  
        validation_fraction=0.1,  
        n_iter_no_change=20  
    ))
])


param_grid = {
    'model__learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'model__max_iter': [100, 200, 300, 500],  
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__min_samples_leaf': [10, 20, 50, 100, 200],  
    'model__l2_regularization': [0.0, 0.01, 0.1, 1.0, 10.0],  
    'model__max_leaf_nodes': [31, 63, 127, 255],  
    'model__max_bins': [128, 255, 512],  
}


random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    n_iter=50,  
    cv=5,  
    scoring='roc_auc',
    n_jobs=4,  
    verbose=2,
    random_state=42
)






In [32]:
best_params = {
    'model__min_samples_leaf': 100,
    'model__max_leaf_nodes': 31,
    'model__max_iter': 300,
    'model__max_depth': 10,
    'model__max_bins': 128,
    'model__learning_rate': 0.05,
    'model__l2_regularization': 1.0
}
pipe.set_params(**best_params)
pipe.fit(X,y)
y_prod_fit=pipe.predict_proba(X)[:, 1]
roc_auc_score(y,y_prod_fit)

0.8409650172275887

In [33]:
# param_grid = {
#     'model__learning_rate': [0.01, 0.05, 0.1],
#     'model__n_estimators': [300, 500, 800],
#     'model__max_depth': [2, 3, 4],
#     'model__subsample': [0.6, 0.8, 1.0],
#     'model__max_features': ['sqrt', 'log2']
# }
# num_features = [
#     'loan_amount_last','overdraft_limit_min',
#     'overdraft_limit_max','offered_rate',
#     'cb_rate','cnt_deb_ul_ip_90','balance_rur_amt_30_min',
#     'cnt_cred_loan_90','fl_hdb_bki_total_active_products','count_all_corp_dashboard_events'
# ]

# cat_features = [
#     'fl_adminarea','db_group_last'
# ]
# flag_features = [
#     'balance_rur_amt_30_min_flag',
#     'cnt_cred_loan_90_flag'
# ]

# X = df_feature[num_features + cat_features + flag_features]
# y = df['target_value']

# preprocess = ColumnTransformer([
#     ('num', StandardScaler(), num_features),
#     ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
# ])

# pipe = Pipeline([
#     ('prep', preprocess),
#     ('model', HistGradientBoostingClassifier(
#  loss='log_loss',
#  max_depth=3,
#  learning_rate=0.2
#  ))
# ])
# grid = RandomizedSearchCV(
#     pipe,
#     param_distributions=param_grid,
#     cv=5,
#     scoring='roc_auc',
#     n_jobs=3,
#     verbose=2
# )


# pipe.fit(X, y)
# y_prod_fit=pipe.predict_proba(X)[:, 1]
# roc_auc_score(y,y_prod_fit)

In [34]:
# subsample=0.6,
#  n_estimators=300,
#  max_features='log2',
#  max_depth=2,
#  learning_rate=0.01

In [35]:
X

,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,cnt_deb_ul_ip_90,balance_rur_amt_30_min,cnt_cred_loan_90,fl_hdb_bki_total_active_products,count_all_corp_dashboard_events,fl_adminarea,db_group_last,balance_rur_amt_30_min_flag,cnt_cred_loan_90_flag
0,1.339991,-1.847954,-1.586546,1.774424,-0.400695,0.147288,-5.561599,0.000000,0.000000,0.000000,NaN,lombard,0,1
1,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,-4.466656,1.466815,0.000000,0.000000,-3.400318,NaN,inn_scoring,0,0
2,2.185431,3.167063,2.369547,-0.709770,-0.400695,5.291707,4.041974,0.000000,0.000000,3.015012,NaN,NaN,0,0
3,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,0.147288,-3.322507,0.000000,0.000000,0.000000,NaN,NaN,0,1
4,0.845440,4.559196,3.467730,-2.484194,-0.400695,-1.373655,4.324315,6.729738,0.000000,1.506380,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145236,1.339991,2.480726,1.828130,3.903733,1.602779,2.599715,-5.527199,0.000000,1.302069,1.589065,Новосибирская область,zalog_light,0,0
145237,0.845440,3.639458,3.605443,1.596982,1.602779,-1.037194,3.052725,0.000000,-4.700790,0.665258,г. Москва,inn_scoring,0,0
145238,1.690880,3.960429,2.995393,3.903733,1.602779,2.863267,-5.561599,8.491984,-1.351879,2.272156,г. Санкт - Петербург,inn_scoring,0,0
145239,0.845440,-2.742346,-2.292086,2.661636,1.602779,0.238218,-5.561599,16.983968,1.674455,-0.305594,Красноярский край,inn_scoring,0,0


In [36]:
del df_feature



In [37]:
df_feature=pd.read_csv('test_apps.csv',sep=',')
idx=df_feature['front_id']

In [38]:
df_feature['cnt_deb_ul_ip_90'] = df_feature['cnt_deb_ul_ip_90'].fillna(df_feature['cnt_deb_ul_ip_90'].median())
df_feature['cnt_cred_loan_90_flag']=df_feature['cnt_cred_loan_90'].isna().astype(int)
df_feature['cnt_cred_loan_90'] = df_feature['cnt_cred_loan_90'].fillna(df_feature['cnt_cred_loan_90'].median())
df_feature['fl_hdb_bki_total_active_products'] = df_feature['fl_hdb_bki_total_active_products'].fillna(df_feature['fl_hdb_bki_total_active_products'].median())
df_feature['balance_rur_amt_30_min_flag']=df_feature['balance_rur_amt_30_min'].isna().astype(int)
df_feature['balance_rur_amt_30_min']=df_feature['balance_rur_amt_30_min'].fillna(0)
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] < -50,-9999)
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] > 50,9999)
df_feature['count_all_corp_dashboard_events'] = df_feature['count_all_corp_dashboard_events'].fillna(0)
df_feature=df_feature.drop(columns=['front_id','decision_day',
                                    'corp_credit_products','sum_deb_ul_90',
                                    'sum_deb_ul_30','cnt_deb_loan_90',
                                    'sum_deb_ul_90','cnt_deb_ul_ip_30',
                                    'loan_rev_max_start_non_fin','loan_rev_min_start_fin',
                                    'app_term_mean_360','overdraft_app_term_max_360','days_from_authperson_registration',
                                    'corp_list',
                                    'p75_time_spent_minutes','sum_deb_investment_90'],axis=1)

In [41]:

df_out = pd.DataFrame()
df_out['front_id'] = idx
df_out['target_value'] = pipe.predict_proba(df_feature)[:, 1]
df_out.to_csv('submission.csv', index=False)